# 00_2 — Account Data

**Purpose:** Load Account Data JSON exports (library, playlists, user attributes, inferences, search queries) and build reference parquets. Enrich `playlist_links.parquet` with playlist classification and build `track_dates.parquet`.

**Runs after:** `00_1_build_full_history.ipynb` (cell 8 reads `playlist_links.parquet` created in `00_1`).

## Inputs / Outputs

| Input | Path | Format |
|---|---|---|
| YourLibrary | `data/Spotify/Spotify_Account_ Data/**/YourLibrary.json` | JSON |
| Playlists | `data/Spotify/Spotify_Account_ Data/**/Playlist*.json` | JSON |
| User attributes | `data/Spotify/Spotify_Account_ Data/**/UserAttributes.json` | JSON |
| Identity | `data/Spotify/Spotify_Account_ Data/**/Identity.json` | JSON |
| Inferences | `data/Spotify/Spotify_Account_ Data/**/Inferences.json` | JSON |
| Search queries | `data/Spotify/Spotify_Account_ Data/**/SearchQueries.json` | JSON |
| Liked songs (dates) | `data/Exportify/**/Liked_Songs.csv` | CSV |
| `playlist_links` | `data/00_output/playlist_links.parquet` | Parquet |

| Output parquet | Description |
|---|---|
| `library_tracks.parquet` | Liked tracks |
| `playlists_metadata.parquet` | Playlist metadata + items |
| `user_attributes.parquet` | Account attributes |
| `inferences.parquet` | Spotify inferences |
| `search_queries.parquet` | Search history |
| `owner_ids.parquet` | Account owner IDs + display names |
| `playlist_links.parquet` | Enriched with `name_norm`, `playlist_type`, `has_external_adders` |
| `track_dates.parquet` | First library / first playlist added dates |

## Limitations

- **N=1** — one profile only, numbers don't transfer.
- **Requires `00_1` first** — cell 8 reads `playlist_links.parquet`, which is created in `00_1`.
- **Missing JSON files are skipped silently** — not all accounts have `Inferences.json` or `SearchQueries.json`. The notebook doesn't fail, it just produces fewer rows.
- **Account detection is automatic** — subdirectories = multiple accounts, root = one account. Directory name is used as `account` fallback when `UserAttributes.json` has no `email`/`username`.
- **`track_dates` may be sparse** — `library_added_at` only for tracks exported in `Liked_Songs.csv`, `first_playlist_added_at` only for tracks added by `owner_ids`. Outer join → many rows with only one date.

## Cell map

| # | Section |
|---|---------|
| 1 | Imports, paths, account detection |
| 2 | `YourLibrary.json` → `library_tracks.parquet` |
| 3 | `Playlist*.json` → `playlists_metadata.parquet` |
| 4 | `UserAttributes.json` → `user_attributes.parquet` |
| 5 | `Inferences.json` → `inferences.parquet` |
| 6 | `SearchQueries.json` → `search_queries.parquet` |
| 7 | Build `owner_ids.parquet` |
| 8 | Classify playlists in `playlist_links.parquet` |
| 9 | Build `track_dates.parquet` |

## 1. Imports, paths, account detection

In [1]:
# 1. Imports, paths, account detection

import sys
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd

# --- Make src.* importable from any working directory ---
_root = Path.cwd()
while not (_root / 'src').is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.paths import (
    get_data_path,
    get_base_path,
    get_project_root,
)

# --- Timezone offset (0 = keep UTC) ---
UTC_OFFSET_HOURS = 0

# --- Paths ---
PROJECT_ROOT = get_project_root()
DATA_DIR = get_data_path()
ACCOUNT_DATA_DIR = DATA_DIR / 'Spotify' / 'Spotify_Account_ Data'
BASE_PATH = get_base_path()

# --- Detect account directories ---
# 1) multiple accounts: ACCOUNT_DATA_DIR / <account> / *.json
# 2) single account:    ACCOUNT_DATA_DIR / *.json
json_files = list(ACCOUNT_DATA_DIR.glob('*.json'))
if json_files:
    account_dirs = [ACCOUNT_DATA_DIR]
else:
    account_dirs = [p.parent for p in ACCOUNT_DATA_DIR.glob('*/*.json')]

account_dirs = sorted(set(account_dirs))

if not account_dirs:
    raise FileNotFoundError(f'No JSON files found under {ACCOUNT_DATA_DIR}')

# --- Summary ---
print(f'UTC offset:      {UTC_OFFSET_HOURS}')
print(f'Account dirs:    {len(account_dirs)}')
print(f'UTC offset:      {UTC_OFFSET_HOURS}')
print(f'Account dirs:    {len(account_dirs)}')

UTC offset:      0
Account dirs:    3
UTC offset:      0
Account dirs:    3


## 2. YourLibrary.json → library_tracks.parquet

In [2]:
# 2. Load YourLibrary.json -> library_tracks.parquet

all_tracks = []

for account_dir in account_dirs:
    json_path = account_dir / 'YourLibrary.json'
    if not json_path.exists():
        continue

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as uf:
                user_data = json.load(uf)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    for item in data.get('tracks', []):
        all_tracks.append({
            'spotify_track_uri': item.get('uri', ''),
            'track_name': item.get('track', ''),
            'artist_name': item.get('artist', ''),
            'album_name': item.get('album', ''),
            'account': account,
        })

df_library = pd.DataFrame(all_tracks)

df_library = df_library[
    df_library['spotify_track_uri'].notna() &
    (df_library['spotify_track_uri'].str.strip() != '')
].copy()

df_library['is_own_library'] = True
df_library['added_at'] = pd.NaT

df_library.to_parquet(BASE_PATH / 'library_tracks.parquet', index=False)

print(f'Library rows:     {len(df_library)}')
print(f'Unique URIs:      {df_library["spotify_track_uri"].nunique()}')
print(f'Accounts:         {df_library["account"].nunique()}')
print(f'Saved:            library_tracks.parquet')

print(df_library.dtypes)

Library rows:     2567
Unique URIs:      2358
Accounts:         3
Saved:            library_tracks.parquet
spotify_track_uri            object
track_name                   object
artist_name                  object
album_name                   object
account                      object
is_own_library                 bool
added_at             datetime64[ns]
dtype: object


## 3. Playlist*.json → playlists_metadata.parquet

In [3]:
# 3. Load Playlist*.json -> playlists_metadata.parquet

EXCLUDED_PLAYLISTS = {'Orphan', 'Liked_Songs'}

def normalize_name(name):
    return name.lower().strip().replace(' ', '_').replace('*', '').replace('"', '').strip('_')

# --- Map account_dir -> spotify_id (username from UserAttributes.json) ---
account_to_spotify_id = {}
for account_dir in account_dirs:
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_attrs = json.load(f)
            username = user_attrs.get('username')
            if username:
                account_to_spotify_id[account_dir] = username
        except Exception:
            pass

playlist_data = []

for account_dir in account_dirs:
    # Account: email/username from UserAttributes, fallback to dir name
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_data = json.load(f)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    playlist_files = list(account_dir.glob('Playlist*.json'))
    for json_path in playlist_files:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for pl in data.get('playlists', []):
            name = pl.get('name', '')
            if name in EXCLUDED_PLAYLISTS:
                continue

            last_modified = pl.get('lastModifiedDate', '')
            items = pl.get('items', [])
            num_tracks = len(items)
            collaborators = pl.get('collaborators', [])
            collaborative = bool(collaborators)
            number_of_followers = pl.get('numberOfFollowers', 0)
            owner_id = account_to_spotify_id.get(account_dir, account)

            for item in items:
                track_info = item.get('track', {}) or {}
                playlist_data.append({
                    'account': account,
                    'playlist_name': name,
                    'name_norm': normalize_name(name),
                    'last_modified': last_modified,
                    'num_tracks': num_tracks,
                    'spotify_track_uri': track_info.get('trackUri', ''),
                    'track_name': track_info.get('trackName', ''),
                    'artist_name': track_info.get('artistName', ''),
                    'album_name': track_info.get('albumName', ''),
                    'owner_id': owner_id,
                    'collaborative': collaborative,
                    'collaborators': json.dumps(collaborators, ensure_ascii=False),
                    'number_of_followers': number_of_followers,
                })

df_playlists = pd.DataFrame(playlist_data)

# --- Drop rows with empty URI ---
df_playlists = df_playlists[
    df_playlists['spotify_track_uri'].notna() &
    (df_playlists['spotify_track_uri'].str.strip() != '')
].copy()

# --- Save ---
df_playlists.to_parquet(BASE_PATH / 'playlists_metadata.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_playlists)}')
print(f'Unique playlists:  {df_playlists["playlist_name"].nunique()}')
print(f'Accounts:          {df_playlists["account"].nunique()}')
print(f'Saved:             playlists_metadata.parquet')

print(df_playlists.dtypes)

Rows:              13417
Unique playlists:  157
Accounts:          3
Saved:             playlists_metadata.parquet
account                object
playlist_name          object
name_norm              object
last_modified          object
num_tracks              int64
spotify_track_uri      object
track_name             object
artist_name            object
album_name             object
owner_id               object
collaborative            bool
collaborators          object
number_of_followers     int64
dtype: object


## 4. UserAttributes.json → user_attributes.parquet

In [4]:
# 4. Load UserAttributes.json -> user_attributes.parquet

user_info = []

for account_dir in account_dirs:
    user_attrs_path = account_dir / 'UserAttributes.json'
    if not user_attrs_path.exists():
        continue

    try:
        with open(user_attrs_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        continue

    account = data.get('email') or data.get('username') or account_dir.name

    user_info.append({
        'account': account,
        'email': data.get('email', ''),
        'country': data.get('country', ''),
        'language': data.get('language', ''),
        'birthdate': data.get('birthdate', ''),
        'gender': data.get('gender', ''),
    })

df_user = pd.DataFrame(user_info)
df_user.to_parquet(BASE_PATH / 'user_attributes.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_user)}')
print(f'Columns:           {list(df_user.columns)}')
print(f'Saved:             user_attributes.parquet')

print(df_user.dtypes)

Rows:              3
Columns:           ['account', 'email', 'country', 'language', 'birthdate', 'gender']
Saved:             user_attributes.parquet
account      object
email        object
country      object
language     object
birthdate    object
gender       object
dtype: object


## 5. Inferences.json → inferences.parquet

In [5]:
# 5. Load Inferences.json -> inferences.parquet

inference_list = []

for account_dir in account_dirs:
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_data = json.load(f)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    json_path = account_dir / 'Inferences.json'
    if not json_path.exists():
        continue

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        continue

    for value in data.get('inferences', []):
        inference_list.append({
            'account': account,
            'value': value,
        })

df_inf = pd.DataFrame(inference_list)
df_inf.to_parquet(BASE_PATH / 'inferences.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_inf)}')
print(f'Accounts:          {df_inf["account"].nunique() if len(df_inf) else 0}')
print(f'Saved:             inferences.parquet')

print(df_inf.dtypes)

Rows:              104
Accounts:          2
Saved:             inferences.parquet
account    object
value      object
dtype: object


## 6. SearchQueries.json → search_queries.parquet

In [6]:
# 6. Load SearchQueries.json -> search_queries.parquet

search_list = []

for account_dir in account_dirs:
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_data = json.load(f)
            account = user_data.get('email') or user_data.get('username') or account_dir.name
        except Exception:
            pass

    json_path = account_dir / 'SearchQueries.json'
    if not json_path.exists():
        continue

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        continue

    if isinstance(data, list):
        queries = data
    else:
        queries = data.get('searchQueries', [])

    for q in queries:
        if isinstance(q, dict):
            search_list.append({
                'account': account,
                'query': q.get('searchQuery', ''),
                'date': q.get('searchInteractionDate', ''),
            })
        else:
            search_list.append({
                'account': account,
                'query': q,
                'date': None,
            })

df_search = pd.DataFrame(search_list)
df_search.to_parquet(BASE_PATH / 'search_queries.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(df_search)}')
print(f'Accounts:          {df_search["account"].nunique() if len(df_search) else 0}')
print(f'Saved:             search_queries.parquet')

print(df_search.dtypes)

Rows:              13
Accounts:          1
Saved:             search_queries.parquet
account    object
query      object
date       object
dtype: object


## 7. Build owner_ids.parquet

In [7]:
# 7. Build owner_ids.parquet

rows = []

skipped_no_username = 0

for account_dir in account_dirs:
    account = account_dir.name
    user_attrs_path = account_dir / 'UserAttributes.json'
    identity_path = account_dir / 'Identity.json'

    username = None
    display_name = account

    if user_attrs_path.exists():
        try:
            with open(user_attrs_path, 'r', encoding='utf-8') as f:
                user_attrs = json.load(f)
            username = user_attrs.get('username')
        except Exception:
            pass

    if identity_path.exists():
        try:
            with open(identity_path, 'r', encoding='utf-8') as f:
                identity = json.load(f)
            display_name = identity.get('displayName', display_name)
        except Exception:
            pass

    if username:
        rows.append({
            'owner_id': username,
            'display_name': display_name,
        })
    else:
        skipped_no_username += 1

owner_ids_df = pd.DataFrame(rows).drop_duplicates('owner_id')
owner_ids_df.to_parquet(BASE_PATH / 'owner_ids.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(owner_ids_df)}')
print(f'Columns:           {list(owner_ids_df.columns)}')
print(f'Saved:             owner_ids.parquet')
if skipped_no_username:
    print(f'⚠️ Rows skipped (no username): {skipped_no_username}')
print(owner_ids_df.dtypes)

Rows:              3
Columns:           ['owner_id', 'display_name']
Saved:             owner_ids.parquet
owner_id        object
display_name    object
dtype: object


## 8. Classify playlists in playlist_links.parquet

In [8]:
# 8. Classify playlists in playlist_links.parquet

playlist_links = pd.read_parquet(BASE_PATH / 'playlist_links.parquet')
playlists_metadata = pd.read_parquet(BASE_PATH / 'playlists_metadata.parquet')
owner_ids = set(pd.read_parquet(BASE_PATH / 'owner_ids.parquet')['owner_id'].tolist())

# --- Clean empty URIs ---
playlist_links = playlist_links[
    playlist_links['Track URI'].notna() &
    (playlist_links['Track URI'].str.strip() != '')
].copy()

# --- Add name_norm ---
playlist_links['name_norm'] = playlist_links['playlist_name'].apply(normalize_name)

if 'name_norm' not in playlists_metadata.columns:
    playlists_metadata['name_norm'] = playlists_metadata['playlist_name'].apply(normalize_name)

# --- Drop technical duplicates ---
before = len(playlist_links)
playlist_links = playlist_links.drop_duplicates(
    subset=['name_norm', 'Track URI', 'Added By', 'Added At'],
    keep='first'
)
dropped_dup = before - len(playlist_links)

# --- Personal / collaborative sets ---
personal_names = set(playlists_metadata[~playlists_metadata['collaborative']]['name_norm'].unique())
collaborative_names = set(playlists_metadata[playlists_metadata['collaborative']]['name_norm'].unique())

# --- Adders per playlist ---
adders_map = playlist_links.groupby('name_norm')['Added By'].apply(set).to_dict()

def classify_playlist(name_norm):
    if name_norm in personal_names:
        return 'personal'
    elif name_norm in collaborative_names:
        return 'collaborative'
    else:
        adders = adders_map.get(name_norm, set())
        if any(ad in owner_ids for ad in adders if pd.notna(ad)):
            return 'contributed'
        else:
            return 'foreign'

playlist_links['playlist_type'] = playlist_links['name_norm'].apply(classify_playlist)

def has_external_adders(name_norm):
    adders = adders_map.get(name_norm, set())
    for ad in adders:
        if pd.isna(ad):
            continue
        if ad == 'unknown':
            continue
        if ad not in owner_ids:
            return True
    return False

playlist_links['has_external_adders'] = playlist_links['name_norm'].apply(has_external_adders)

# --- Save ---
playlist_links.to_parquet(BASE_PATH / 'playlist_links.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(playlist_links)}')
print(f'Dropped (dupes):   {dropped_dup}')
print(f'Saved:             playlist_links.parquet')
print(f'\nplaylist_type:')
print(playlist_links['playlist_type'].value_counts().to_string())

print(playlist_links.dtypes)

Rows:              33094
Dropped (dupes):   7553
Saved:             playlist_links.parquet

playlist_type:
playlist_type
foreign          19534
personal         10028
collaborative     3443
contributed         89
Track URI              object
Added By               object
Added At               object
playlist_name          object
account                object
name_norm              object
playlist_type          object
has_external_adders      bool
dtype: object


## 9. Build track_dates.parquet

In [9]:
# 9. Build track_dates.parquet

EXPORTIFY_DIR = DATA_DIR / 'Exportify'

# --- 1. Library-added dates from Liked_Songs.csv ---
liked_csv_files = list(EXPORTIFY_DIR.rglob('Liked_Songs.csv'))
if liked_csv_files:
    liked_dfs = []
    for f in liked_csv_files:
        df_liked = pd.read_csv(f)
        liked_dfs.append(df_liked[['Track URI', 'Added At']])
    liked_dates_all = pd.concat(liked_dfs, ignore_index=True)
    liked_dates_all['Added At'] = (
        pd.to_datetime(liked_dates_all['Added At'], errors='coerce')
        + pd.Timedelta(hours=UTC_OFFSET_HOURS)
    )
    liked_dates_all = liked_dates_all.dropna(subset=['Added At'])
    library_added_dates = (
        liked_dates_all.groupby('Track URI')['Added At'].min().reset_index()
    )
    library_added_dates.columns = ['spotify_track_uri', 'library_added_at']
else:
    library_added_dates = pd.DataFrame(columns=['spotify_track_uri', 'library_added_at'])

# --- 2. First-added-to-own-playlist dates ---
pl = pd.read_parquet(BASE_PATH / 'playlist_links.parquet')
OWNER_IDS = set(pd.read_parquet(BASE_PATH / 'owner_ids.parquet')['owner_id'].tolist())

my_pl = pl[pl['Added By'].isin(OWNER_IDS)]
EXCLUDED_PLAYLISTS_NORM = {'liked_songs', 'orphan'}
my_pl = my_pl[~my_pl['name_norm'].isin(EXCLUDED_PLAYLISTS_NORM)]

playlist_added_dates = (
    my_pl.groupby('Track URI')['Added At'].min().reset_index()
)
playlist_added_dates['Added At'] = (
    pd.to_datetime(playlist_added_dates['Added At'], errors='coerce')
    + pd.Timedelta(hours=UTC_OFFSET_HOURS)
)
playlist_added_dates = playlist_added_dates.dropna(subset=['Added At'])
playlist_added_dates.columns = ['spotify_track_uri', 'first_playlist_added_at']

# --- 3. Merge both date sources ---
track_dates = pd.merge(
    library_added_dates, playlist_added_dates,
    on='spotify_track_uri', how='outer'
)

# --- Save ---
track_dates.to_parquet(BASE_PATH / 'track_dates.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(track_dates)}')
print(f'With library date: {int(track_dates["library_added_at"].notna().sum())}')
print(f'With playlist date:{int(track_dates["first_playlist_added_at"].notna().sum())}')
print(f'Both dates:        {int((track_dates["library_added_at"].notna() & track_dates["first_playlist_added_at"].notna()).sum())}')
print(f'Saved:             track_dates.parquet')

print(track_dates.dtypes)

Rows:              10315
With library date: 2358
With playlist date:9641
Both dates:        1684
Saved:             track_dates.parquet
spotify_track_uri                       object
library_added_at           datetime64[ns, UTC]
first_playlist_added_at    datetime64[ns, UTC]
dtype: object


<details>
<summary>🇷🇺 Итог</summary>

# 00_2 — Account Data

**Что делает:** Парсит JSON-выгрузку Spotify Account Data (библиотека, плейлисты, атрибуты, инференсы, поисковые запросы) и собирает справочные паркеты. Обогащает `playlist_links.parquet` классификацией плейлистов, собирает `track_dates.parquet`.

**Запускается после:** `00_1_build_full_history.ipynb` (ячейка 8 читает `playlist_links.parquet`, созданный в `00_1`).

## Входы / Выходы

| Вход | Путь |
|---|---|
| YourLibrary | `data/Spotify/Spotify_Account_ Data/**/YourLibrary.json` |
| Playlists | `data/Spotify/Spotify_Account_ Data/**/Playlist*.json` |
| User attributes | `data/Spotify/Spotify_Account_ Data/**/UserAttributes.json` |
| Identity | `data/Spotify/Spotify_Account_ Data/**/Identity.json` |
| Inferences | `data/Spotify/Spotify_Account_ Data/**/Inferences.json` |
| Search queries | `data/Spotify/Spotify_Account_ Data/**/SearchQueries.json` |
| Liked songs (даты) | `data/Exportify/**/Liked_Songs.csv` |
| `playlist_links` | `data/00_output/playlist_links.parquet` |

| Выход (parquet) | Что внутри |
|---|---|
| `library_tracks.parquet` | Лайки |
| `playlists_metadata.parquet` | Метаданные плейлистов + items |
| `user_attributes.parquet` | Атрибуты аккаунта |
| `inferences.parquet` | Инференсы Spotify |
| `search_queries.parquet` | История поиска |
| `owner_ids.parquet` | Владельцы аккаунтов + display names |
| `playlist_links.parquet` | Обогащён `name_norm`, `playlist_type`, `has_external_adders` |
| `track_dates.parquet` | Даты добавления в библиотеку / плейлист |

## Ограничения

- **N=1** — проверено на одном профиле, числа не переносятся.
- **Требует запуска `00_1`** — ячейка 8 читает `playlist_links.parquet`, созданный в `00_1`.
- **Отсутствующие JSON пропускаются молча** — не у всех аккаунтов есть `Inferences.json` или `SearchQueries.json`. Ноут не падает, просто получается меньше строк.
- **Определение аккаунтов автоматическое** — подпапки = несколько аккаунтов, корень = один. Имя директории используется как `account`, если в `UserAttributes.json` нет `email`/`username`.
- **`track_dates` может быть разреженным** — `library_added_at` только для треков из `Liked_Songs.csv`, `first_playlist_added_at` только для добавленных `owner_ids`. Outer join → много строк с одной датой.

## Карта ячеек

| # | Секция |
|---|--------|
| 1 | Импорты, пути, определение аккаунтов |
| 2 | `YourLibrary.json` → `library_tracks.parquet` |
| 3 | `Playlist*.json` → `playlists_metadata.parquet` |
| 4 | `UserAttributes.json` → `user_attributes.parquet` |
| 5 | `Inferences.json` → `inferences.parquet` |
| 6 | `SearchQueries.json` → `search_queries.parquet` |
| 7 | Сборка `owner_ids.parquet` |
| 8 | Классификация плейлистов в `playlist_links.parquet` |
| 9 | Сборка `track_dates.parquet` |

</details>